# Künstliche Intelligenz für unseren Online-Shop (Mit ECHTEN Daten)

Jetzt wird es ernst! Wir nutzen den berühmten **Online Shoppers Purchasing Intention Dataset** (12.330 echte Shop-Besuche).
Die KI soll wieder vorhersagen: **Kauft der Besucher (Revenue = True) oder nicht (Revenue = False)?**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# 1. Echte Daten laden
df = pd.read_csv("online_shoppers_intention.csv")

print("So sehen die ECHTEN Daten aus (erste 5 Besucher):")
display(df.head())

print("\nVerteilung von Käufern und Nicht-Käufern im Datensatz:")
print(df['Revenue'].value_counts())

So sehen die ECHTEN Daten aus (erste 5 Besucher):


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



Verteilung von Käufern und Nicht-Käufern im Datensatz:
Revenue
False    10422
True      1908
Name: count, dtype: int64


### Daten anpassen (Features auswählen und Skalieren)
Der Datensatz hat Text-Spalten (z.B. Monat = 'Feb'). Das mag die KI (ohne fortgeschrittene Umwandlung) nicht. Wir machen es uns für den Anfang leicht und nehmen nur die messbaren (numerischen) Spalten wie 'Dauer' und 'Absprungraten' (BounceRates).

In [2]:
# 2. Daten aufteilen und anpassen
# Wir suchen uns einige spannende numerische Spalten aus
numerische_spalten = [
    'Administrative', 'Administrative_Duration', 
    'Informational', 'Informational_Duration', 
    'ProductRelated', 'ProductRelated_Duration', 
    'BounceRates', 'ExitRates', 'PageValues'
]

X = df[numerische_spalten]

# Unser Ziel (Target): Hat der User gekauft?
# Das ist True oder False. Wir wandeln das für die KI in 1 (Kauf) und 0 (Kein Kauf) um.
y = df['Revenue'].astype(int)

# Wir nehmen 80% der Daten zum Lernen (Training) und 20% zum Abfragen (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Skalieren (Zahlen für die KI 'glätten', extrem wichtig bei KNN!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"{X_train_scaled.shape[0]} Besucher werden genutzt, damit die KI lernt.")
print(f"{X_test_scaled.shape[0]} Besucher heben wir auf, um die KI am Ende zu testen.")

9864 Besucher werden genutzt, damit die KI lernt.
2466 Besucher heben wir auf, um die KI am Ende zu testen.


### Das künstliche Gehirn trainieren
Dieses Mal machen wir das 'Gehirn' ein winziges bisschen größer, da echte Daten komplexer sind als unsere ausgedachten Daten.

In [3]:
# 3. KI trainieren
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8), # Etwas größer als vorher
    activation='relu',
    max_iter=1000, 
    random_state=42
)

print("Die KI lernt jetzt aus fast 10.000 echten Besuchern... Bitte warten...")
mlp.fit(X_train_scaled, y_train)
print("Fertig! Die KI hat die Muster gelernt.")

Die KI lernt jetzt aus fast 10.000 echten Besuchern... Bitte warten...
Fertig! Die KI hat die Muster gelernt.


### Die Prüfung: Wie gut ist die KI auf ECHTEN Daten?
Hier kommt die Wahrheit ans Licht. Schaffst die KI es, in einem echten Online-Shop Käufer von Nicht-Käufern zu unterscheiden?

In [4]:
# 4. Prüfung der Vorhersagen
from sklearn.metrics import confusion_matrix

vorhersagen = mlp.predict(X_test_scaled)
matrix = confusion_matrix(y_test, vorhersagen)
wahr_nein, falsch_ja, falsch_nein, wahr_ja = matrix.ravel()

print("=== ZEUGNIS FÜR DIE KI AUF ECHTEN DATEN ===\n")
print(f"Wir haben {len(y_test)} echte Besucher getestet.\n")

print("HIER HAT DIE KI RECHT GEHABT:")
print(f"✔️ {wahr_nein} Besucher haben nicht gekauft. Die KI wusste das.")
print(f"✔️ {wahr_ja} Besucher haben gekauft. Die KI hat es korrekt vorhergesehen.\n")

print("HIER HAT SICH DIE KI GEIRRT:")
print(f"❌ {falsch_ja} Fehlalarme: Die KI dachte, sie kaufen. Haben sie aber NICHT.")
print(f"❌ {falsch_nein} Übersehen: Sie haben gekauft, aber die KI dachte, sie tun es NICHT.\n")

genauigkeit = ((wahr_nein + wahr_ja) / len(y_test)) * 100
print("=== ZUSAMMENFASSUNG ===")
print(f"Die KI lag in {genauigkeit:.1f} % der Fälle richtig!")

=== ZEUGNIS FÜR DIE KI AUF ECHTEN DATEN ===

Wir haben 2466 echte Besucher getestet.

HIER HAT DIE KI RECHT GEHABT:
✔️ 1972 Besucher haben nicht gekauft. Die KI wusste das.
✔️ 210 Besucher haben gekauft. Die KI hat es korrekt vorhergesehen.

HIER HAT SICH DIE KI GEIRRT:
❌ 83 Fehlalarme: Die KI dachte, sie kaufen. Haben sie aber NICHT.
❌ 201 Übersehen: Sie haben gekauft, aber die KI dachte, sie tun es NICHT.

=== ZUSAMMENFASSUNG ===
Die KI lag in 88.5 % der Fälle richtig!
